# VoiceStudio on free Colab - Pro-style remote GPU

Your PC runs ONLY the UI. Every bit of CPU/GPU work happens here on the T4.
This is what "Pro" sells (remote GPU + all features + multiple devices), free.

## First time
1. Runtime -> Change runtime type -> **T4 GPU**
2. Runtime -> **Run all** (or run boxes 1, 2, 3 by hand)
3. Scroll to the END: the **URL** and **API key** are printed there.
4. On the PC: double-click the **VoiceStudio** icon, paste the URL, then
   paste the key (the launcher puts it on your clipboard).

Box 4 reprints the pair. Box 5 adds dubbing (needs your Hugging Face token).

## When Colab restarts
* Tab reconnect / kernel restart -> run **box 2** only, paste the new URL.
  (Same key: it lives on Google Drive, so your PC stays paired.)
* Fresh VM -> run **box 1** then **box 2**. Models are already on Drive.

## What is stored where
| What | Where |
|---|---|
| Models, settings, cloned voices | Google Drive (`/content/drive/omnivoice`) |
| API key | Google Drive - survives VM resets |
| Public URL | changes every fresh tunnel (that is the security) |
| Compute | Colab T4 only. Your PC does nothing. |

In [ ]:
# 1. Install the app on Colab (run once per VM; ~5-10 min)
INSTALL_GPU_TURBO = True   # ~2x faster generations. Set False to skip the extra download.
PINNED_COMMIT = "f47221a1f0af89df27c34747471daa30391a9487"  # tested with this notebook
# Keep models, settings and your API KEY on Google Drive, so when Colab recycles
# the VM (it will) everything is still there: no 5 GB re-download, and the key
# already saved on your PC keeps working - only the URL changes.
PERSIST_TO_GOOGLE_DRIVE = True

import importlib.metadata as _md, importlib.util, json, os, shutil, subprocess, sys, time

print("VoiceStudio setup: box 1/6 - install the app on this VM.")
print("Takes 5-10 minutes the first time. Do not close this tab.")

def run(cmd, cwd=None):
    print(f"\n$ {cmd if isinstance(cmd, str) else ' '.join(cmd)}", flush=True)
    rc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode
    if rc != 0:
        raise SystemExit(f"FAILED ({rc}). Re-run this box.")

# --- GPU check -------------------------------------------------------------
if not shutil.which("nvidia-smi"):
    raise SystemExit("No GPU found. Runtime -> Change runtime type -> T4 GPU, then re-run.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv"], capture_output=True, text=True).stdout.strip())

REPO_DIR = "/content/VoiceStudio"
REPO_URL = "https://github.com/debpalash/VoiceStudio"
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

# --- Clone (pinned so this notebook always matches the code it targets) -----
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    for attempt in (1, 2, 3):        # Colab->GitHub drops connections often
        rc = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                            capture_output=True, text=True).returncode
        if rc == 0:
            break
        print(f"git clone attempt {attempt}/3 failed, retrying in 5s...")
        subprocess.run("rm -rf " + REPO_DIR, shell=True)
        time.sleep(5)
    else:
        raise SystemExit("Could not download the app from GitHub (network). Re-run this box.")
_head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR,
                       capture_output=True, text=True).stdout.strip()
if _head != PINNED_COMMIT and not os.environ.get("COLAB_UNPINNED"):
    run(["git", "fetch", "--depth", "1", "origin", PINNED_COMMIT], cwd=REPO_DIR)
    run(["git", "checkout", "-q", PINNED_COMMIT], cwd=REPO_DIR)
    _head = PINNED_COMMIT
print("Repo at", _head[:12])

# --- ffmpeg (dubbing/decoding) --------------------------------------------
if shutil.which("ffmpeg"):
    print("ffmpeg present.")
else:
    for attempt in range(1, 4):
        if subprocess.run("apt-get update -qq", shell=True).returncode == 0:
            break
        print(f"apt-get update retry {attempt}/3 in 5s...")
        time.sleep(5)
    run("apt-get install -y -qq --fix-missing ffmpeg libsndfile1")

# --- App + engine dependencies --------------------------------------------
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
for attempt in (1, 2):              # a 10-minute step must not die on one hiccup
    rc = subprocess.run(["uv", "pip", "install", "--system", "--no-cache",
                         "--constraint", "deploy/torch-constraints.txt", "."],
                        cwd=REPO_DIR).returncode
    if rc == 0:
        break
    print(f"dependency install attempt {attempt}/2 failed - retrying...")
    time.sleep(10)
else:
    raise SystemExit("Dependency install failed twice. Re-run this box.")
os.makedirs(os.path.join(REPO_DIR, ".venv"), exist_ok=True)
run([sys.executable, os.path.join(REPO_DIR, "scripts", "setup.py")])
run([sys.executable, "-c",
     "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"],
    cwd=REPO_DIR)

# --- GPU Turbo (optional; never allowed to break the install) -------------
if INSTALL_GPU_TURBO and importlib.util.find_spec("flashinfer") is None:
    try:
        import torch as _torch_probe
        _cu = (_torch_probe.version.cuda or "12.8").replace(".", "")   # "12.8" -> "128"
        _idx = f"https://flashinfer.ai/whl/cu{_cu}"
        print(f"Installing GPU Turbo (FlashInfer) from {_idx} ...")
        _pin = "/content/turbo-constraints.txt"
        with open(_pin, "w") as _f:
            _f.write(f"torch=={_md.version('torch')}\n")
            _f.write(f"torchaudio=={_md.version('torchaudio')}\n")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                        "-c", _pin, "flashinfer-python", "flashinfer-jit-cache",
                        "--extra-index-url", _idx], check=False)
        # A broken install must never slow generations: import-test it, and
        # remove the package outright if it cannot even load.
        if importlib.util.find_spec("flashinfer") is not None:
            subprocess.run([sys.executable, "-c", "import flashinfer"], check=True)
        else:
            raise RuntimeError("no flashinfer module after install")
    except Exception as exc:  # noqa: BLE001 - optional feature, never fatal
        print("GPU Turbo not available (%s). Continuing without it." % str(exc)[:160])
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "flashinfer-python", "flashinfer-jit-cache"], check=False)
if PERSIST_TO_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:  # noqa: BLE001 - install is fine without Drive
        print("Drive not mounted (%s)." % str(exc)[:120])
        print("Continuing WITHOUT persistence: a Colab reset will mean a new key")
        print("and re-downloaded models.")
        PERSIST_TO_GOOGLE_DRIVE = False

if PERSIST_TO_GOOGLE_DRIVE:
    root = "/content/drive/omnivoice"
    os.makedirs(root + "/hf", exist_ok=True)
    os.makedirs(root + "/data", exist_ok=True)    # settings, licences, cloned voices

    # The HF cache normally stores weights as symlinks into blobs/, which
    # Drive's FUSE mount refuses. HF_HUB_DISABLE_SYMLINKS=1 makes huggingface_hub
    # COPY the files instead, while keeping the same snapshots/ layout that the
    # app's is_cached() detector understands - so the cache CAN live on Drive.
    # The flag is read once at import time, so it is set before anything imports
    # huggingface_hub.
    os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"
    hf_dir = root + "/hf"
    os.makedirs(hf_dir, exist_ok=True)
    with open(root + "/config.json", "w") as fh:
        json.dump({"hf": hf_dir, "data": root + "/data", "root": root,
                   "models_on_drive": hf_dir == root + "/hf"}, fh, indent=1)
    os.environ["HF_HOME"] = hf_dir
    print("Drive persistence ON at", root)
    print("  settings/voices on Drive:", root + "/data")
    print("  model cache            :", hf_dir,
          "(copy mode: Drive FUSE cannot hold symlinks, so HF copies instead)")

print("Box 1 done.")

In [ ]:
# 2. Start the backend + public link (run after box 1; prints the URL + KEY for your PC)
#    Re-run this box any time Colab restarts the app or you want a fresh pair.
NEW_TUNNEL = False  # True only if you deliberately want a brand-new public link

import importlib.util, json, os, re, secrets, socket, subprocess, sys, threading, time, urllib.error, urllib.request

REPO_DIR = "/content/VoiceStudio"
PORT = 3900
LOG = "/content/omnivoice_backend.log"
KEY_FILE = "/content/omnivoice_api_key.txt"
URL_FILE = "/content/last_tunnel_url.txt"
LOCAL = f"http://127.0.0.1:{PORT}"

# Persisted state from box 1 (survives a Colab reset). The mount is per kernel
# session, so remount here too - otherwise the key would silently regenerate and
# the one already saved on your PC would stop working.
DRIVE_ROOT = "/content/drive/omnivoice"
DRIVE = {}
if not os.path.isdir("/content/drive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        print("Drive remounted for this session.")
    except Exception:  # noqa: BLE001 - not fatal, just no persistence
        pass
try:
    if os.path.exists(os.path.join(DRIVE_ROOT, "config.json")):
        DRIVE = json.load(open(os.path.join(DRIVE_ROOT, "config.json")))
        os.environ["HF_HOME"] = DRIVE["hf"]
        print("Drive persistence active:", DRIVE["hf"])
    elif os.path.isdir("/content/drive"):
        print("NOTE: Drive is mounted but box 1 persistence was never set up.")
except Exception as exc:  # noqa: BLE001 - fall back to a normal run
    print("Drive state unreadable (%s); running without it." % str(exc)[:120])
    DRIVE = {}

# Clear failure instead of a cryptic Popen error when the app is not installed.
if not os.path.isdir(os.path.join(REPO_DIR, "backend")):
    raise SystemExit("The app is not installed in this VM yet. Run box 1 first.")

def health(url, timeout=6):
    """True when /health reports ok. Never raises."""
    ok, _reason = health_detail(url, timeout)
    return ok

def health_detail(url, timeout=6):
    """(ok, reason) - a swallowed exception used to look identical for DNS
    failures, TLS errors, a booting backend and a dead tunnel."""
    try:
        with urllib.request.urlopen(url + "/health", timeout=timeout) as r:
            if r.status != 200:
                return False, f"HTTP {r.status} (backend still starting?)"
            body = json.load(r)
            if body.get("status") == "ok":
                return True, "answered"
            return False, f"answered but status={body.get('status')!r}"
    except urllib.error.HTTPError as exc:
        return False, f"HTTP {exc.code}"
    except urllib.error.URLError as exc:
        reason = getattr(exc, "reason", exc)
        if isinstance(reason, socket.timeout):
            return False, "timed out - tunnel or network problem"
        return False, f"cannot reach it ({reason})"
    except Exception as exc:  # noqa: BLE001
        return False, f"{type(exc).__name__}: {str(exc)[:80]}"

def exchange(url, key, timeout=20):
    """Exactly what the desktop app does (electron remote-backend.ts)."""
    req = urllib.request.Request(
        url + "/api/auth/session",
        data=json.dumps({"transport": "bearer"}).encode(),
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {key}"},
        method="POST")
    payload = json.load(urllib.request.urlopen(req, timeout=timeout))
    token = str(payload.get("token", ""))
    assert re.fullmatch(r"ovs_admin_session_[A-Za-z0-9_-]{43}", token), payload
    return payload

def _free_port(port):
    """Free the backend port without depending on lsof.

    Minimal images (Colab among them) often ship without lsof, where
    `kill $(lsof -t -i:PORT)` silently does nothing and the stale backend keeps
    the port - the replacement then cannot bind at all. Match our own uvicorn by
    reading /proc, then fall back to fuser for anything else on the socket.
    """
    import signal
    killed = []
    for entry in os.listdir("/proc"):
        if not entry.isdigit():
            continue
        try:
            with open("/proc/%s/cmdline" % entry, "rb") as fh:
                argv = fh.read().decode("utf8", "replace").split("\0")
        except OSError:
            continue
        try:
            ours = ("uvicorn" in argv and "--port" in argv
                    and argv[argv.index("--port") + 1] == str(port))
        except (ValueError, IndexError):
            ours = False
        if ours:
            try:
                os.kill(int(entry), signal.SIGKILL)
                killed.append(entry)
            except OSError:
                pass
    subprocess.run("fuser -k %s/tcp 2>/dev/null || true" % port, shell=True)
    if killed:
        print("Freed port %s from stale backend(s): %s" % (port, ", ".join(killed)))


def start_backend(key):
    env = os.environ.copy()
    env.pop("OMNIVOICE_API_KEY", None)
    env["OMNIVOICE_API_KEY"] = key
    env["OMNIVOICE_SERVER_MODE"] = "1"
    env["OMNIVOICE_DATA_DIR"] = DRIVE.get("data", "/content/omnivoice_data")
    if "hf" in DRIVE:
        env["HF_HOME"] = DRIVE["hf"]
    env["OMNIVOICE_ALLOWED_HOSTS"] = ".trycloudflare.com"
    env["PYTHONUNBUFFERED"] = "1"
    if DRIVE:
        # Same copy mode as box 1, so the backend reads the same Drive cache.
        env["HF_HUB_DISABLE_SYMLINKS"] = "1"
    if importlib.util.find_spec("flashinfer") is not None:
        env["OMNIVOICE_FLASHINFER"] = "graph"   # CUDA graphs: best for one clip at a time
        print("GPU Turbo: FlashInfer active.")
    else:
        env.pop("OMNIVOICE_FLASHINFER", None)
    _free_port(PORT)
    time.sleep(2)
    open(LOG, "w").close()
    log = open(LOG, "ab")
    return subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", "backend",
         "--host", "127.0.0.1", "--port", str(PORT)],
        cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)

def wait_health(seconds=300):
    deadline = time.time() + seconds
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(LOCAL + "/health", timeout=5) as r:
                return json.load(r)
        except Exception:
            time.sleep(3)
    raise SystemExit("Backend did not come up. Run: !tail -40 /content/omnivoice_backend.log")

# --- The API key: reused across restarts ---------------------------------
# The desktop app keeps only a short-lived session in memory, never the key,
# so a stable key means one paste keeps working after restarts.
KEY = ""
for _path in ([os.path.join(DRIVE_ROOT, "api_key.txt")] if DRIVE else []) + [KEY_FILE]:
    try:
        _candidate = open(_path).read().strip()
    except OSError:
        continue
    if len(_candidate) >= 32:
        KEY = _candidate
        break
if len(KEY) < 32:
    KEY = secrets.token_urlsafe(32)
open(KEY_FILE, "w").write(KEY)
if DRIVE:
    # The key outlives the VM, so the one saved on your PC stays valid.
    with open(os.path.join(DRIVE_ROOT, "api_key.txt"), "w") as _fh:
        _fh.write(KEY)
    print("API key stored on Drive - survives Colab resets.")

proc = start_backend(KEY)
info = wait_health()
try:
    exchange(LOCAL, KEY)
except Exception:
    # A previous backend was still holding the port: start clean with a new key.
    print("Old backend still held the port - restarting clean with a new key.")
    KEY = secrets.token_urlsafe(32)
    open(KEY_FILE, "w").write(KEY)
    if DRIVE:
        with open(os.path.join(DRIVE_ROOT, "api_key.txt"), "w") as _fh:
            _fh.write(KEY)
    proc = start_backend(KEY)
    wait_health()
    exchange(LOCAL, KEY)
print("Backend ready:", info)

# --- Public link: reuse it when it is genuinely alive ----------------------
url = ""
if not NEW_TUNNEL and os.path.exists(URL_FILE):
    candidate = open(URL_FILE).read().strip()
    tunnel_alive = subprocess.run("pgrep -x cloudflared",
                                  shell=True).returncode == 0
    if candidate and tunnel_alive and health(candidate, timeout=8):
        url = candidate
        print("Reusing the live link from the previous run.")
if not url:
    subprocess.run("pkill -x cloudflared || true", shell=True)
    for attempt in (1, 2):
        subprocess.run("wget -q -O /usr/local/bin/cloudflared "
                       "https://github.com/cloudflare/cloudflared/releases/latest/download/"
                       "cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared",
                       shell=True, check=False)
        if os.path.isfile("/usr/local/bin/cloudflared") and os.access("/usr/local/bin/cloudflared", os.X_OK):
            break
        print(f"cloudflared download attempt {attempt} failed - retrying...")
        time.sleep(5)
    else:
        raise SystemExit("Could not download cloudflared (GitHub download blocked?).\n"
                         "Re-run this box in a minute, or open a new Colab tab and retry.")
    open("/content/cloudflared.log", "w").close()
    subprocess.Popen("nohup cloudflared tunnel --url http://127.0.0.1:3900 "
                     "--no-autoupdate > /content/cloudflared.log 2>&1 &", shell=True)
    for _ in range(30):
        time.sleep(3)
        found = re.search(r"https://[^\s]*trycloudflare\.com",
                          open("/content/cloudflared.log", errors="replace").read())
        if found and health(found.group(0)):
            url = found.group(0)
            break
    if not url:
        log = open("/content/cloudflared.log", errors="replace").read()
        if "trycloudflare.com" not in log:
            raise SystemExit("cloudflared started but never handed out a link.\n"
                             "This is a Colab/network hiccup - re-run this box in a minute.")
        last = [ln for ln in log.splitlines() if ln.strip()][-3:]
        ok, why = health_detail(re.search(r"https://[^\s]*trycloudflare\.com", log).group(0), 10)
        raise SystemExit("The public link did not answer: " + why + "\n"
                         + "\n".join(last))
    open(URL_FILE, "w").write(url)

# --- Prove the pair works from OUTSIDE, exactly as the PC app will --------
session = exchange(url, KEY, timeout=25)
req = urllib.request.Request(url + "/system/info",
                             headers={"Authorization": f"Bearer {session['token']}"})
with urllib.request.urlopen(req, timeout=20) as r:
    assert isinstance(json.load(r).get("app_version"), str)
print("Public link verified end to end.")

# --- Keep Colab awake while you work --------------------------------------
def _heartbeat():
    while True:
        time.sleep(120)
        print("[alive]", time.strftime("%H:%M:%S"), "- keep this tab open", flush=True)
threading.Thread(target=_heartbeat, daemon=True).start()

print("=" * 66)
print("COPY BOTH - from this same run, below:")
print("=" * 66)
print("URL:\n" + url)
print("\nAPI KEY:\n" + KEY)
print("\nOn the PC: double-click the VoiceStudio icon, paste the URL,")
print("then paste the KEY (Ctrl+V). With both, every button works.")
print("=" * 66)

In [ ]:
# 3. One-time app setup: models, licences, privacy, Max quality (run after box 2)
#    Safe to re-run: downloads are cached and every step is idempotent.

import json, os, urllib.error, urllib.request

BASE = "http://127.0.0.1:3900"
DRIVE_ROOT = "/content/drive/omnivoice"

def call(path, body=None, method=None):
    req = urllib.request.Request(
        BASE + path,
        data=None if body is None else json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
        method=method or ("PUT" if body is not None else "GET"))
    with urllib.request.urlopen(req, timeout=90) as r:
        raw = r.read().decode()
    return json.loads(raw) if raw.strip() else {}

try:
    call("/health")
except Exception:
    raise SystemExit("Backend is not running - run box 2 first.")

# --- Re-apply the persistence config from box 1 ---------------------------
# Without this, a kernel restart loses HF_HOME and the models would download
# into a cache the backend never looks at - the app would then report every
# model as "still needed" even though the download succeeded.
CACHE = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
if not os.path.isdir("/content/drive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        pass
try:
    cfg = json.load(open(os.path.join(DRIVE_ROOT, "config.json")))
    if cfg.get("hf"):
        CACHE = cfg["hf"]
except Exception:
    pass
os.environ["HF_HOME"] = CACHE
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS", "1")   # read at import time
print("model cache:", CACHE)

# --- 1) Models -------------------------------------------------------------
# Installs are detected straight from this cache, so downloading here is
# identical to clicking Install in the app.
from huggingface_hub import snapshot_download
MODELS = [
    "k2-fsa/OmniVoice",                                  # voice + TTS
    "eustlb/higgs-audio-v2-tokenizer",                   # TTS tokenizer
    "Systran/faster-whisper-large-v3",                   # transcription
    "csukuangfj/sherpa-onnx-whisper-tiny",               # dictation
    "csukuangfj/sherpa-onnx-nemo-parakeet-tdt-0.6b-v3-int8",
    "facebook/nllb-200-distilled-600M",                  # translation
]
failed = []
for repo in MODELS:
    try:
        snapshot_download(repo)
        print(f"  ready: {repo}")
    except Exception as exc:  # keep going: one bad repo must not hide the rest
        failed.append(repo)
        print(f"  FAILED: {repo} - {type(exc).__name__}: {str(exc)[:120]}")
if failed:
    print("")
    print("Some downloads failed (network?). Re-run this box - finished ones are cached.")

# --- 2) Licence acceptance (YOUR decision) --------------------------------
# Read each licence in the app's Model Catalogue first. This records that YOU
# accepted the terms; it is deliberately NOT pre-approved, because acceptance is
# a legal acknowledgement and it would be wrong to record it for you.
ACCEPT_MODEL_LICENCES = False
if ACCEPT_MODEL_LICENCES:
    for repo in ["k2-fsa/OmniVoice", "eustlb/higgs-audio-v2-tokenizer"]:
        details = call(f"/models/licenses/details/{repo}")
        fp = (details.get("acceptance") or {}).get("fingerprint")
        if not fp:
            print(f"  {repo}: no fingerprint published - skipping")
            continue
        call("/models/licenses/accept",
             {"repo_id": repo, "fingerprint": fp, "accepted": True},
             method="POST")
        print(f"  licence accepted: {repo}")
else:
    print("licences: NOT accepted yet.")
    print("  Read them in the app under Model Catalogue, then set")
    print("  ACCEPT_MODEL_LICENCES = True above and re-run this box.")

# --- 3) Privacy choice -----------------------------------------------------
SHARE_ANONYMOUS_STATS = False   # False = nothing is ever sent
state = call("/api/settings/analytics", {"enabled": bool(SHARE_ANONYMOUS_STATS)}, "PUT")
print("  analytics:", json.dumps(state))

# --- 4) Max quality profile ------------------------------------------------
try:
    profile = call("/api/settings/performance-profile", {"tier": "max"})
except urllib.error.HTTPError as exc:
    if exc.code == 409:
        raise SystemExit("A job is still running - finish it, then re-run this box.")
    raise
tier = profile.get("global")
tts = (profile.get("targets") or {}).get("tts") or {}
print("  tier:", tier, "| TTS:", json.dumps(tts))

# Verify, don't assume. A silent "SETUP FINISHED" over a tier that never
# applied is worse than a clear failure: every generation would quietly run at
# the default profile instead of Max.
problems = list(failed)
if tier != "max":
    problems.append("the Max tier did not apply (backend returned %r)" % tier)
elif tts.get("steps") != 64:
    problems.append("Max applied but TTS steps are %r, expected 64" % tts.get("steps"))

print("")
if problems:
    print("=" * 66)
    print("SETUP INCOMPLETE - fix these, then re-run this box:")
    for item in problems:
        print("  -", item)
    print("=" * 66)
    raise SystemExit(1)

print("=" * 66)
print("SETUP FINISHED.")
print("On the PC: double-click the VoiceStudio icon,")
print("paste the URL from box 2, then paste the API key (Ctrl+V).")
print("=" * 66)


In [ ]:
# 4. Reprint your URL + key (any time you lose them; app must still be running)
#    Free to run any time. Does NOT restart anything.
# Prefer the Drive copy of the key: that is the one that outlived the VM and
# therefore the one already saved on your PC. Fall back to this VM's copy.
def _read(*paths):
    for path in paths:
        try:
            value = open(path).read().strip()
        except OSError:
            continue
        if value:
            return value
    return ""

key = _read("/content/drive/omnivoice/api_key.txt", "/content/omnivoice_api_key.txt")
url = _read("/content/last_tunnel_url.txt")
if not url or not key:
    print("No saved pair here - run box 2 first.")
else:
    print("URL:\n" + url)
    print("\nAPI KEY:\n" + key)
    print("\nDouble-click the VoiceStudio icon on your PC and paste both.")

In [ ]:
# 5. OPTIONAL: dubbing pack - speaker separation (pyannote)
#    Needed only for Dubbing (and multi-speaker transcripts).
#
#    This one needs YOU, once, on the web:
#      1. Create a free account at https://huggingface.co and log in.
#      2. Open https://huggingface.co/pyannote/speaker-diarization-3.1
#         and click the green "Agree and access repository" button.
#      3. Open https://huggingface.co/pyannote/segmentation-3.0
#         and click the same button.
#      4. Go to https://huggingface.co/settings/tokens ->
#         "Create new token" -> pick the READ role -> copy it.
#    Paste the token below (it stays on this VM, it is never sent anywhere else).

HF_TOKEN = ""

import importlib.util, json, os, re, subprocess, sys, urllib.request

BASE = "http://127.0.0.1:3900"
PYANNOTE_REPOS = ["pyannote/speaker-diarization-3.1", "pyannote/segmentation-3.0",
                  "pyannote/wespeaker-voxceleb-resnet34-LM"]

def call(path, body=None, method=None):
    req = urllib.request.Request(
        BASE + path,
        data=None if body is None else json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
        method=method or ("PUT" if body is not None else "GET"))
    with urllib.request.urlopen(req, timeout=90) as r:
        raw = r.read().decode()
    return json.loads(raw) if raw.strip() else {}

# Point downloads at exactly the cache the backend reads. Without this, a
# kernel restart silently sends them to ~/.cache while the backend looks in
# Drive, and dubbing then reports the models as missing.
DRIVE_ROOT = "/content/drive/omnivoice"
if not os.path.isdir("/content/drive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        pass
try:
    _cfg = json.load(open(os.path.join(DRIVE_ROOT, "config.json")))
    if _cfg.get("hf"):
        os.environ["HF_HOME"] = _cfg["hf"]
except Exception:
    pass
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS", "1")
print("model cache:", os.environ.get("HF_HOME", "default"))

token = (HF_TOKEN or "").strip()
if not token.startswith("hf_"):
    raise SystemExit("Paste your Hugging Face READ token above (hf_...) and re-run this box.")
os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACE_HUB_TOKEN"] = token

print("Saving the token to the backend ...")
print("  ", json.dumps(call("/api/settings/hf-token", {"token": token}, "POST"))[:200])

from huggingface_hub import snapshot_download
for repo in PYANNOTE_REPOS:
    snapshot_download(repo)          # small: config + embedding model
    print(f"  ready: {repo}")

for repo in PYANNOTE_REPOS:          # licence acceptance, same as box 3
    details = json.dumps(call(f"/models/licenses/details/{repo}"))
    match = re.search(r"v2:[a-f0-9]{64}", details)
    if match:
        call("/models/licenses/accept",
             {"repo_id": repo, "fingerprint": match.group(0), "accepted": True},
             method="POST")
        print(f"  licence accepted: {repo}")

if importlib.util.find_spec("pyannote") is None:   # pinned so torch never changes
    # Pin to the EXACT installed versions. A loose floor like torch>=2.8 would
    # let pip pull a different torch build over the one box 1 installed, and the
    # compiled CUDA stack would stop matching it.
    import importlib.metadata as _md
    pin = "/content/diarization-constraints.txt"
    with open(pin, "w") as fh:
        for _pkg in ("torch", "torchaudio"):
            try:
                fh.write(f"{_pkg}=={_md.version(_pkg)}\n")
            except _md.PackageNotFoundError:
                pass
    print("  pinned:", open(pin).read().split())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                    "-c", pin, "pyannote.audio"], check=False)
if importlib.util.find_spec("pyannote") is None:
    raise SystemExit("pyannote.audio could not be installed - share the error text.")

profile = call("/api/settings/performance-profile", {"tier": "max"})
print("  tier:", profile.get("global"),
      "| diarisation:", profile.get("selections", {}).get("diarisation", {}).get("engine"))
print("Box 4 done - Dubbing is ready (first run loads the model: ~1 min).")